# Проект спринта 27. Часть 1: анализ и подготовка данных

Это первая из двух тетрадок, в которых вы будете выполнять проект по извлечению именованных сущностей (NER). Здесь вы познакомитесь с разметкой WikiANN, исследуете русскоязычную часть датасета и подготовите её для BERT: токенизируете предложения и выровняете NER-метки
с подтокенами.

**Датасет:** [WikiANN](https://huggingface.co/datasets/unimelb-nlp/wikiann), русскоязычная
конфигурация. Типы сущностей: `PER`, `ORG`, `LOC`. Схема разметки — IOB2.

**Токенизатор:** `bert-base-multilingual-cased`.

> **Где выполнять:** на своём компьютере — GPU здесь не нужен. Не запускайте виртуальную машину, она понадобится только во второй части проекта.

## Как работать с тетрадкой

Выполняйте ячейки последовательно. Дополняйте ячейки с пометкой `TODO`, сохраняйте
результаты вычислений и записывайте выводы в соответствующих ячейках Markdown.

Структура тетрадки:

- Шаг 1 — подготовка окружения и обеспечение воспроизводимости.
- Шаг 2 — знакомство с данными и EDA.
- Шаг 3 — токенизация и выравнивание меток.
- Написание выводов и самопроверка по чек-листу.

## Шаг 1. Подготовка окружения и обеспечение воспроизводимости

Установите зависимости, импортируйте библиотеки и зафиксируйте `SEED`.

In [ ]:
# Установка библиотек
%pip install -q datasets transformers seqeval spacy

In [ ]:
from collections import Counter
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from seqeval.metrics.sequence_labeling import get_entities
from spacy import displacy
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
    set_seed,
)

In [ ]:
SEED = 42
set_seed(SEED)

print("Версии: " + ", ".join(
    f"{package}={version(package)}"
    for package in [
        "torch",
        "transformers",
        "datasets",
        "seqeval",
        "spacy",
    ]
))

## Шаг 2. Знакомство и анализ данных

WikiANN построен автоматически на основе статей Википедии. Один пример содержит
исходные `tokens`, числовые `ner_tags` и служебные поля `langs`, `spans`.

В IOB2 каждая новая сущность начинается с `B-*`, её продолжение размечается `I-*`,
а токены вне сущностей — `O`. При подсчёте распределения одна метка `B-*`
соответствует одной сущности.


In [ ]:
# Загрузка русскоязычной конфигурации WikiANN
raw_datasets = load_dataset("unimelb-nlp/wikiann", "ru")

print(f"Размер обучающей выборки:   {len(raw_datasets['train'])}")
print(f"Размер валидационной:       {len(raw_datasets['validation'])}")
print(f"Размер тестовой:            {len(raw_datasets['test'])}")

# Получаем метки из структуры датасета
label_list = raw_datasets["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}
ENTITY_TYPES = sorted({
    label.split("-")[-1]
    for label in label_list
    if label != "O"
})

print(f"Метки: {label_list}")
print(f"Типы сущностей: {ENTITY_TYPES}")


### Анализ примеров разметки

Выведите первые две именованные сущности из обучающей выборки. Покажите исходный текст и пары «токен — строковая метка» для размеченных сущностей.


In [ ]:
# TODO: дополните вывод двух примеров
for i in range(2):
    tokens = raw_datasets["train"][i]["tokens"]
    tags = [label_list[tag] for tag in raw_datasets["train"][i]["ner_tags"]]

    # Выведите предложение и токены с метками, отличными от O
    # Ваш код:


### Демонстрация сущностей

Используйте готовую функцию и выведите три предложения из валидационной выборки, в которых подсвечены размеченные сущности. Проверьте, что многословные сущности отображаются единым фрагментом.


In [ ]:
def visualize_ner(example, label_list):
    """Подсвечивает целые IOB2-сущности в одном примере WikiANN."""
    tokens = example["tokens"]
    tags = [label_list[tag] for tag in example["ner_tags"]]
    text = " ".join(tokens)

    offsets = []
    cursor = 0
    for token in tokens:
        offsets.append((cursor, cursor + len(token)))
        cursor += len(token) + 1

    ents = [
        {
            "start": offsets[start][0],
            "end": offsets[end][1],
            "label": entity_type,
        }
        for entity_type, start, end in get_entities(tags)
    ]
    displacy.render(
        {"text": text, "ents": ents},
        style="ent",
        manual=True,
        jupyter=True,
    )


# TODO: вызовите visualize_ner для трёх примеров из валидационной выборки


### Анализ распределения сущностей

На обучающей выборке посчитайте количество сущностей `PER`, `ORG`, `LOC` по меткам `B-*`,
выведите полученные значения и отобразите их на столбчатой диаграмме. Не считайте `I-*` отдельными сущностями.


In [ ]:
# TODO: сформируйте список типов сущностей по тегам B-*
# entity_types = ...
# type_counts = ...

# TODO: выведите значения и постройте столбчатую диаграмму


### Изучение длин предложений

Рассчитайте длины предложений в обучающей выборки по числу исходных `tokens`.
Постройте распределение и выведите минимум, максимум и медиану.


In [ ]:
# TODO: рассчитайте длины предложений и постройте гистограмму
# sentence_lengths = ...

# TODO: выведите минимум, максимум и медиану


### Выводы

> **TODO:** кратко опишите соотношение типов сущностей, характерные длины предложений и замеченные особенности автоматической разметки WikiANN.


## Блок 3. Токенизация и выравнивание меток

BERT может разбить одно исходное слово на несколько токенов-подслов. В этом проекте
исходная NER-метка назначается только первому токену. Продолжения слова,
специальные токены и паддинг получают `-100` и исключаются из расчёта функции потерь.


In [ ]:
model_checkpoint = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


In [ ]:
def tokenize_and_align_labels(examples):
    """Токенизирует слова и выравнивает NER-метки с подсловами."""
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


### Анализ работы word_ids()

С помощью зафиксированного `SEED` выберите два примера из обучающей выборки. Для каждой позиции выведите последовательность значений:

```text
токен → word_id → метка после выравнивания
```

Проверьте специальные токены, первый токен слова и его продолжения.


In [ ]:
examples_to_trace = raw_datasets["train"].shuffle(seed=SEED).select(range(2))

for i in range(len(examples_to_trace["tokens"])):
    tokens = examples_to_trace["tokens"][i]
    tags = examples_to_trace["ner_tags"][i]
    tokenized = tokenizer(tokens, is_split_into_words=True)
    word_ids = tokenized.word_ids()
    subtokens = tokenizer.convert_ids_to_tokens(tokenized["input_ids"])

    # TODO: выровняйте метки по тем же правилам, что и в функции выше
    # aligned_labels = ...

    # TODO: выведите токен, word_id и читаемую метку для каждой позиции


### Почему используется `-100`

> **TODO:** объясните, почему специальные токены и продолжения слова получают `-100`,
> а не класс `O` или копию исходной метки.


### Применениние функции ко всему датасету

Примените `tokenize_and_align_labels` к обучающей, валидационной и тестовой выборкам
с помощью `.map()`. Функция рассчитана на обработку нескольких примеров, поэтому
обязательно укажите `batched=True`. Удалите исходные столбцы через
`remove_columns=raw_datasets["train"].column_names`.


In [ ]:
# TODO: примените tokenize_and_align_labels ко всем выборкам
#
# Подсказка по обязательным аргументам:
# tokenized_datasets = raw_datasets.map(
#     tokenize_and_align_labels,
#     batched=True,
#     remove_columns=raw_datasets["train"].column_names,
# )


### Проверка выравнивания

Для каждого примера во всех выборках проверьте равенство длин `input_ids`,
`attention_mask`, `labels`. На первых 100 примерах каждой выборки убедитесь, что
`-100` стоит на специальных токенах и продолжениях слов.


In [ ]:
# TODO: проверьте длины для каждого примера во всех трёх выборках

# TODO: проверьте позиции -100 на первых 100 примерах каждой выборки


### Проверка динамического паддинга

Создайте коллатор и демонстрационный батч из четырёх примеров. Проверьте формы
`input_ids`, `attention_mask`, `labels` и метки на позициях padding.


In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# TODO: сформируйте батч из четырёх обучающих примеров
# TODO: проверьте равенство форм
# TODO: проверьте attention_mask == 0 и labels == -100 на padding

### Вывод

> **TODO:** подтвердите результатами проверок, что все три выборки готовы к подаче в модель. Укажите назначение `word_ids()` и `-100`.


## Что дальше

Подготовка данных завершена. Вторая часть проекта посвящена обучению моделей и анализу ошибок, её нужно выполнять на виртуальной машине с GPU.

Переносить подготовленные данные на VM не требуется: во второй тетрадке есть код для загрузки и подготовки готовки данных без пропущенных ячеек. Разбираться в нём заново
не нужно — вы уже сделали это здесь.

Прежде чем инициализировать VM, убедитесь, что чек-лист ниже выполнен полностью:
исправлять подготовку данных, расходуя ограниченное время работы с GPU, невыгодно.

## Ресурсы

- [WikiANN](https://huggingface.co/datasets/unimelb-nlp/wikiann)
- [Hugging Face Datasets](https://huggingface.co/docs/datasets/)
- [Hugging Face: Token Classification](https://huggingface.co/docs/transformers/tasks/token_classification)
- [bert-base-multilingual-cased](https://huggingface.co/bert-base-multilingual-cased)

## Чек-лист готовности первой части проекта

### Анализ данных

- [ ] Выведены размеры выборок, список меток и типы сущностей.
- [ ] Показаны два примера именованных сущностей с читаемой разметкой.
- [ ] Продемонстрировано минимум три предложения с подсвеченными именованными сущностями.
- [ ] Построено распределение типов сущностей и сделан вывод о дисбалансе.
- [ ] Построено распределение длин предложений, выведены минимум, максимум и медиана.

### Токенизация и выравнивание

- [ ] Разобрана работа `word_ids()` на двух примерах.
- [ ] Подготовлены обучающая, валидационная и тестовая выборки.
- [ ] Для всех примеров совпадают длины `input_ids`, `attention_mask`, `labels`.
- [ ] `-100` стоит на специальных токенах, паддинге и продолжениях слов.
- [ ] Объяснено назначение `-100`.
- [ ] Проверен целочисленный `dtype` поля `labels` в батче.

### Перед переходом ко второй тетрадке

- [ ] Во всех случайных операциях используется единый `SEED`.
- [ ] Все обязательные `TODO` заполнены, временные заглушки удалены.
- [ ] Ядро перезапущено, тетрадка выполнена сверху вниз без ошибок.
- [ ] Результаты вычислений и визуализации сохранены в тетрадке.